# Ridge Regression - Precios de Propiedades en CABA

Implementación de regularización Ridge (L2) desde scratch para predecir el precio de propiedades en CABA.
Extiende el modelo de regresión lineal del notebook 04 agregando penalización sobre los pesos.

**Features (X)**: `metros`, `ambientes`, `banos`, `expensas`  
**Target (y)**: `precio` en USD

## Estructura
1. Setup e imports
2. Datos (carga, split, scaling)
3. Ridge desde scratch (`compute_cost_ridge`, `compute_gradient_ridge`, `gradient_descent`)
4. Tuning de λ sobre validation
5. Evaluación (comparación con baseline del notebook 04)

### 1. Setup/Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### 2. Datos

Mismo pipeline que el notebook 04: carga → split 70/15/15 → z-score normalization fitteada sobre train.

In [ ]:
df = pd.read_csv("../data/processed/zonaprop_clean.csv")

features = ["metros", "ambientes", "banos", "expensas"]
X = df[features].values
y = df["precio"].values

np.random.seed(42)
idx = np.random.permutation(len(X))
X, y = X[idx], y[idx]
n = len(X)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)
X_train, y_train = X[:n_train],              y[:n_train]
X_val,   y_val   = X[n_train:n_train+n_val], y[n_train:n_train+n_val]
X_test,  y_test  = X[n_train+n_val:],        y[n_train+n_val:]

mu    = X_train.mean(axis=0)
sigma = X_train.std(axis=0)
X_train_s = (X_train - mu) / sigma
X_val_s   = (X_val   - mu) / sigma
X_test_s  = (X_test  - mu) / sigma

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

### 3. Regularización Ridge (L2) desde scratch

La regresión lineal minimiza el error sobre los datos de entrenamiento, pero
nada le impide volverse muy complejo para lograrlo — pesos enormes que se ajustan
perfectamente al train pero fallan en datos nuevos. Eso es **overfitting**.

**Ridge** lo resuelve agregando una penalización al costo proporcional al tamaño de los
pesos. El modelo ya no puede hacer los pesos arbitrariamente grandes sin pagar un precio
en la función de costo:

$$J_{ridge}(\mathbf{w}, b) = \underbrace{\frac{1}{2m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)})^2}_{\text{error}} + \underbrace{\frac{\lambda}{2m} \sum_{j=1}^{n} w_j^2}_{\text{penalización}}$$

El parámetro **λ (lambda)** controla el balance entre los dos términos:

- λ = 0 → sin regularización, idéntico a regresión lineal
- λ grande → pesos forzados a acercarse a cero, modelo más simple (riesgo de underfitting)
- λ chico → penalización suave, se parece a sin regularización

> `b` no se regulariza. La convención estándar es penalizar solo los pesos $\mathbf{w}$,
> porque `b` controla el nivel base de la predicción, no la complejidad del modelo.

Al derivar la función de costo modificada, solo cambia el gradiente de $\mathbf{w}$ —
se le suma el término de penalización. El gradiente de `b` queda igual:

$$\frac{\partial J_{ridge}}{\partial \mathbf{w}} = \frac{1}{m} \mathbf{X}^T (\hat{y} - y) + \frac{\lambda}{m} \mathbf{w} \qquad \frac{\partial J_{ridge}}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)})$$

#### 3.1 `compute_cost_ridge`

In [ ]:
def compute_cost_ridge(X, y, w, b, lambda_):
    m = X.shape[0]
    y_hat = X @ w + b
    cost = (1 / (2 * m)) * np.sum((y_hat - y) ** 2)
    penalty = (lambda_ / (2 * m)) * np.sum(w ** 2)
    return cost + penalty

#### 3.2 `compute_gradient_ridge`

#### 3.3 `gradient_descent` con Ridge

### 4. Tuning de λ

### 5. Evaluación